<a href="https://colab.research.google.com/github/KesteHarshada87/Reinforcement_Learning/blob/main/Expt06(Temporal%20Difference%20(TD)%20Learning).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import numpy as np
import matplotlib.pyplot as plt

# ==========================================
# 1. Environment Setup (Simple 4x4 GridWorld)
# ==========================================
class GridWorld:
    def __init__(self, size=4):
        self.size = size
        self.terminal_states = [(0, 0), (size - 1, size - 1)]
        self.actions = [(-1, 0), (1, 0), (0, -1), (0, 1)]  # Up, Down, Left, Right

    def step(self, state, action_idx):
        if state in self.terminal_states:
            return state, 0, True

        dr, dc = self.actions[action_idx]
        next_state = (max(0, min(self.size - 1, state[0] + dr)),
                      max(0, min(self.size - 1, state[1] + dc)))

        reward = -1  # Step penalty
        done = next_state in self.terminal_states
        return next_state, reward, done

    def get_all_states(self):
        return [(r, c) for r in range(self.size) for c in range(self.size)]

# ==========================================
# 2. TD(0) Prediction Algorithm
# ==========================================
def td_zero_prediction(env, num_episodes=500, alpha=0.1, gamma=1.0):
    V = {state: 0.0 for state in env.get_all_states()}

    for _ in range(num_episodes):
        # Start in a random non-terminal state
        non_terminals = [s for s in env.get_all_states() if s not in env.terminal_states]
        state = non_terminals[np.random.choice(len(non_terminals))]

        done = False
        while not done:
            action_idx = np.random.choice(len(env.actions))  # Random Policy
            next_state, reward, done = env.step(state, action_idx)

            # TD(0) Update Step
            td_target = reward + gamma * V[next_state]
            td_error = td_target - V[state]
            V[state] += alpha * td_error

            state = next_state

    return V

# ==========================================
# 3. First-Visit Monte Carlo Prediction
# ==========================================
def monte_carlo_prediction(env, num_episodes=500, gamma=1.0):
    V = {state: 0.0 for state in env.get_all_states()}
    returns = {state: [] for state in env.get_all_states()}

    for _ in range(num_episodes):
        # Generate an episode
        non_terminals = [s for s in env.get_all_states() if s not in env.terminal_states]
        state = non_terminals[np.random.choice(len(non_terminals))]

        episode = []
        done = False
        while not done:
            action_idx = np.random.choice(len(env.actions))
            next_state, reward, done = env.step(state, action_idx)
            episode.append((state, reward))
            state = next_state

        # Process episode backward to compute returns
        G = 0
        visited_states = set()
        for s, r in reversed(episode):
            G = gamma * G + r
            if s not in visited_states:  # First-visit MC
                visited_states.add(s)
                returns[s].append(G)
                V[s] = np.mean(returns[s])

    return V

# ==========================================
# 4. Helper to print value functions
# ==========================================
def print_value_grid(V, size=4):
    grid = np.zeros((size, size))
    for (r, c), val in V.items():
        grid[r, c] = val
    print(np.round(grid, 2))

# ==========================================
# 5. Run & Compare
# ==========================================
if __name__ == "__main__":
    env = GridWorld(size=4)
    EPISODES = 1000

    v_td = td_zero_prediction(env, num_episodes=EPISODES, alpha=0.1)
    v_mc = monte_carlo_prediction(env, num_episodes=EPISODES)

    print("State-Value Function Estimated by TD(0) ---")
    print_value_grid(v_td)

    print("State-Value Function Estimated by Monte Carlo ---")
    print_value_grid(v_mc)

State-Value Function Estimated by TD(0) ---
[[  0.   -11.88 -20.51 -21.86]
 [-14.53 -18.2  -20.02 -19.43]
 [-20.52 -19.67 -17.06 -11.26]
 [-21.83 -20.59 -15.11   0.  ]]
State-Value Function Estimated by Monte Carlo ---
[[  0.    -8.43 -10.49 -12.79]
 [ -7.98  -8.8   -9.38 -10.8 ]
 [-10.9   -9.52  -8.58  -6.87]
 [-13.11 -10.66  -7.88   0.  ]]
